## Hybrid search, reranking i nowoczesny chunking

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mzarnecki/course_llm_agent_apps_with_langchain_and_langgraph/blob/main/notebooks/pl/7_4_Hybrid_search_reranking_chunking.ipynb)

| | |
|---|---|
| 🎯 **Czego się nauczysz** | Wyszukiwanie w produkcyjnym RAG: BM25 + embeddingi łączone przez Reciprocal Rank Fusion, reranking cross-encoderem oraz sześć strategii chunkingu, w tym Contextual Retrieval. |
| 🔑 **Klucze API** | `OPENAI_API_KEY` |
| 💲 **Koszt API** | bardzo niski – embeddingi kilkudziesięciu krótkich tekstów i ~10 krótkich zapytań do LLM |
| ⏱️ **Czas wykonania** | ~1–3 min (pierwsze uruchomienie pobiera model `BAAI/bge-reranker-v2-m3`, ~2,3 GB) |

**Plan:**
1. Hybrid search – BM25 + embeddingi + Reciprocal Rank Fusion (RRF)
2. Reranking z użyciem cross-encodera
3. Nowoczesne strategie chunkingu – fixed, recursive, semantic, structure-aware, parent-child, Contextual Retrieval
4. Wszystko razem – pipeline RAG w stylu produkcyjnym

### Instalacja bibliotek

In [1]:
# Zainstaluj bezpośrednie zależności używane w tym notebooku.
# Pomijane, gdy brak pip, np. w środowisku `uv sync`, które ma je już zainstalowane (zob. README).
import importlib.util
if importlib.util.find_spec("pip"):
    %pip install -qU langchain-openai langchain-community langchain-classic langchain-text-splitters rank-bm25 faiss-cpu sentence-transformers pandas python-dotenv
# Google Colab: dodaj OPENAI_API_KEY w panelu Secrets (🔑) i zezwól notatnikowi na dostęp.
# Odkomentuj te linie, aby wczytać klucz z Colab Secrets do bieżącej sesji:
# from google.colab import userdata
# import os
# os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
# Lokalnie umieść OPENAI_API_KEY=... w pliku .env; nie commituj klucza.

Note: you may need to restart the kernel to use updated packages.


In [2]:
import warnings

from dotenv import load_dotenv
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

# langchain-community wypisuje przy imporcie informację o wygaszaniu pakietu; retrievery nadal działają
warnings.filterwarnings("ignore", category=DeprecationWarning)
load_dotenv()

llm = ChatOpenAI(model="gpt-6-luna")
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

## 1. Hybrid search – BM25 + embeddingi

Wyszukiwanie semantyczne (wektorowe) rozumie **znaczenie**, ale zaskakująco słabo radzi sobie z **dokładnymi tokenami**: kodami błędów, identyfikatorami, numerami umów, kodami produktów. Dla modelu embeddingów `ERR-10532` i `ERR-10533` wyglądają prawie identycznie. Wyszukiwanie leksykalne (BM25) działa odwrotnie: dopasowuje dokładne słowa, ale nie zna synonimów.

| Przykład zapytania | Preferowana metoda |
|---|---|
| „Jak zakończyć umowę?” | Semantic search |
| „Dokument HRB 80430” | Lexical search |
| „Problemy z połączeniem z bazą danych” | Semantic search |
| „Błąd ERR-10532” | Lexical search |
| „Warunki rozwiązania umowy nr 120/2026” | Hybrid search |

Hybrid search uruchamia obie metody i łączy ich rankingi:

```
                 Zapytanie
                     │
          ┌──────────┴──────────┐
          ▼                     ▼
     BM25 Search           Vector Search
          │                     │
          └──────────┬──────────┘
                     ▼
             Rank Fusion (RRF)
                     ▼
             Najlepsze dokumenty
```

Do zajęć nie potrzebujemy Elasticsearcha – wystarczą `rank-bm25`, FAISS i OpenAI Embeddings.

### Baza wiedzy

In [3]:
from langchain_core.documents import Document

knowledge_base = {
    "contract-general": "Zakończenie współpracy: każda ze stron może wypowiedzieć umowę serwisową z 30-dniowym okresem wypowiedzenia, przesłanym e-mailem lub listem.",
    "contract-120": "Umowa nr 120/2026 z Nordic Logistics: rozwiązanie wymaga 90-dniowego wypowiedzenia i rozliczenia wszystkich otwartych faktur.",
    "contract-121": "Umowa nr 121/2026 z Baltic Foods: rozwiązanie wymaga 60-dniowego wypowiedzenia; kaucja jest zwracana w ciągu 14 dni.",
    "db-troubleshooting": "Jeśli aplikacja nie może dostać się do PostgreSQL, sprawdź, czy pula połączeń nie jest wyczerpana i czy firewall przepuszcza ruch na porcie 5432.",
    "db-backup": "Kopie zapasowe bazy danych wykonywane są co noc o 2:00 i przechowywane przez 30 dni.",
    "refunds": "Zwroty płatności kartą realizowane są w ciągu 5–10 dni roboczych od akceptacji.",
    "payment-methods": "Akceptujemy Visa, Mastercard, BLIK i przelewy bankowe. American Express nie jest obsługiwany.",
    "password": "Aby zresetować hasło, kliknij „Nie pamiętam hasła” na stronie logowania i użyj linku wysłanego e-mailem.",
}

# prawdziwe bazy wiedzy zawierają wiele niemal identycznych identyfikatorów
error_messages = [
    "bank odrzucił przelew z powodu braku środków.",
    "wydawca karty wymaga potwierdzenia 3-D Secure przez klienta.",
    "bramka płatności odrzuciła transakcję, ponieważ zapisany token karty wygasł. Poproś klienta o ponowne podanie danych karty.",
    "przekroczono czas generowania faktury PDF. Ponów eksport za kilka minut.",
    "waluta zamówienia nie jest obsługiwana przez konto sprzedawcy.",
    "adres rozliczeniowy nie zgadza się z kartą.",
    "nie udało się odnowić subskrypcji, ponieważ plan został zarchiwizowany.",
    "wykryto zduplikowaną płatność; drugie obciążenie zostało zablokowane.",
]
for code, message in enumerate(error_messages, start=10530):
    knowledge_base[f"err-{code}"] = f"ERR-{code}: {message}"

companies = [
    "ABC Software GmbH, siedziba w Monachium, prezes zarządu Anna Weber",
    "ABC Hardware GmbH, siedziba w Augsburgu, prezes zarządu Jonas Klein",
    "Delta Consulting GmbH, siedziba w Monachium, prezes zarządu Petra Lang",
    "Nord Energie GmbH, siedziba w Hamburgu, prezes zarządu Lukas Brandt",
    "Alpen Bau GmbH, siedziba w Rosenheim, prezes zarządu Maria Huber",
    "Rhein Logistik GmbH, siedziba w Kolonii, prezes zarządu Tim Becker",
]
for number, company in enumerate(companies, start=80428):
    knowledge_base[f"hrb-{number}"] = f"Wpis do rejestru handlowego HRB {number}: {company}."

docs = [Document(page_content=text, metadata={"id": doc_id}) for doc_id, text in knowledge_base.items()]
len(docs)

22

### BM25 – intuicja

BM25 ocenia dokument, sumując dla każdego słowa z zapytania trzy idee:

$$\text{BM25}(q, d) = \sum_{t \in q} \underbrace{\text{IDF}(t)}_{\text{rzadkie = ważne}} \cdot \underbrace{\frac{f(t,d)\,(k_1+1)}{f(t,d) + k_1\,(1 - b + b\,\frac{|d|}{\text{avgdl}})}}_{\text{częstość słowa, z nasyceniem i normalizacją długości}}$$

1. **IDF** – słowo występujące w niewielu dokumentach (`err-10532`) jest warte dużo więcej niż popularne (`i`, `w`).
2. **Nasycenie TF** ($k_1 \approx 1.5$) – dziesiąte wystąpienie słowa dodaje dużo mniej niż pierwsze.
3. **Normalizacja długości** ($b \approx 0.75$) – trafienie w krótkim dokumencie liczy się bardziej niż w długim.

`BM25Retriever` z LangChain opakowuje bibliotekę `rank-bm25`. Tokenizator ma duże znaczenie: domyślny dzieli tekst tylko po spacjach, więc używamy wyrażenia regularnego, które zachowuje identyfikatory takie jak `err-10532` czy `120/2026` jako pojedyncze tokeny.

⚠️ BM25 nie zna polskiej fleksji: „umowę” i „umowy” to dla niego różne słowa. W produkcji stosuje się lematyzację/stemming (np. analizator języka polskiego w Elasticsearch/OpenSearch) – tu tę lukę wypełni wyszukiwanie wektorowe.

In [4]:
import re

from langchain_community.retrievers import BM25Retriever


def tokenize(text: str) -> list[str]:
    # zachowaj identyfikatory takie jak "err-10532" czy "120/2026" jako pojedyncze tokeny
    return re.findall(r"\w+(?:[-/]\w+)*", text.lower())


print(tokenize("Błąd ERR-10532 w umowie nr 120/2026"))

bm25 = BM25Retriever.from_documents(docs, preprocess_func=tokenize, k=5)

# IDF: rzadkie słowa ważą dużo więcej niż popularne
for term in ["err-10532", "wypowiedzenia", "i"]:
    print(f"IDF({term!r:16}) = {bm25.vectorizer.idf.get(term, 0):.2f}")

['błąd', 'err-10532', 'w', 'umowie', 'nr', '120/2026']
IDF('err-10532'     ) = 2.66
IDF('wypowiedzenia' ) = 1.72
IDF('i'             ) = 1.16


### Wyszukiwanie wektorowe (FAISS)

In [5]:
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.from_documents(docs, embeddings)
vector = vectorstore.as_retriever(search_kwargs={"k": 5})

### Leksykalne vs semantyczne – porównanie

Dla każdego zapytania wiemy, który dokument jest poprawną odpowiedzią. ✅ oznacza, że retriever umieścił go na pierwszym miejscu.

In [6]:
import pandas as pd

pd.set_option("display.max_colwidth", 80)

# zapytanie -> id dokumentu, który na nie odpowiada
queries = {
    "Jak zakończyć umowę?": "contract-general",
    "Dokument HRB 80430": "hrb-80430",
    "Problemy z połączeniem z bazą danych": "db-troubleshooting",
    "Błąd ERR-10532": "err-10532",
    "Warunki rozwiązania umowy nr 120/2026": "contract-120",
}


def top_ids(retriever, query: str) -> list[str]:
    return [d.metadata["id"] for d in retriever.invoke(query)]


def compare(retrievers: dict, queries: dict[str, str]) -> pd.DataFrame:
    rows = []
    for query, expected in queries.items():
        row = {"zapytanie": query}
        for name, retriever in retrievers.items():
            top = top_ids(retriever, query)[0]
            row[name] = ("✅ " if top == expected else "❌ ") + top
        rows.append(row)
    return pd.DataFrame(rows)


compare({"BM25": bm25, "Vector": vector}, queries)

,zapytanie,BM25,Vector
0,Jak zakończyć umowę?,✅ contract-general,✅ contract-general
1,Dokument HRB 80430,✅ hrb-80430,❌ hrb-80428
2,Problemy z połączeniem z bazą danych,❌ err-10530,✅ db-troubleshooting
3,Błąd ERR-10532,✅ err-10532,✅ err-10532
4,Warunki rozwiązania umowy nr 120/2026,✅ contract-120,✅ contract-120


### Reciprocal Rank Fusion (RRF)

Wyniki BM25 (0–20+) i podobieństwa kosinusowe (0–1) są w innych skalach, więc nie można ich po prostu dodać. **RRF ignoruje wartości wyników i korzysta tylko z pozycji** w każdym rankingu:

$$\text{RRF}(d) = \sum_{r \in \text{rankingi}} \frac{1}{k + \text{pozycja}_r(d)}, \qquad k = 60$$

Dokument wysoko na *obu* listach wygrywa z dokumentem, który jest pierwszy tylko na jednej z nich. Stała $k$ łagodzi przewagę pierwszych pozycji. To tylko kilka linii kodu:

In [7]:
from collections import defaultdict


def reciprocal_rank_fusion(rankings: list[list[str]], k: int = 60) -> list[tuple[str, float]]:
    scores = defaultdict(float)
    for ranking in rankings:
        for rank, doc_id in enumerate(ranking, start=1):
            scores[doc_id] += 1 / (k + rank)
    return sorted(scores.items(), key=lambda item: item[1], reverse=True)


query = "Warunki rozwiązania umowy nr 120/2026"
bm25_ranking, vector_ranking = top_ids(bm25, query), top_ids(vector, query)
print("BM25:  ", bm25_ranking)
print("Vector:", vector_ranking)
print("RRF:")
for doc_id, score in reciprocal_rank_fusion([bm25_ranking, vector_ranking]):
    print(f"  {score:.4f}  {doc_id}")

BM25:   ['contract-120', 'contract-121', 'hrb-80431', 'hrb-80430', 'hrb-80433']
Vector: ['contract-120', 'contract-121', 'contract-general', 'err-10534', 'err-10537']
RRF:
  0.0328  contract-120
  0.0323  contract-121
  0.0159  hrb-80431
  0.0159  contract-general
  0.0156  hrb-80430
  0.0156  err-10534
  0.0154  hrb-80433
  0.0154  err-10537


In [8]:
# autotest: dokument drugi na obu listach wygrywa z dokumentem pierwszym tylko na jednej
assert reciprocal_rank_fusion([["a", "b"], ["c", "b"]])[0][0] == "b"

### Hybrid retriever w LangChain – `EnsembleRetriever`

`EnsembleRetriever` implementuje dokładnie ten ważony RRF (domyślnie `c=60`). Parametr `weights` pozwala faworyzować jedną z metod, np. `[0.3, 0.7]`, gdy przeważają pytania w języku naturalnym.

In [9]:
from langchain_classic.retrievers import EnsembleRetriever

hybrid = EnsembleRetriever(retrievers=[bm25, vector], weights=[0.5, 0.5])

compare({"BM25": bm25, "Vector": vector, "Hybrid (RRF)": hybrid}, queries)

,zapytanie,BM25,Vector,Hybrid (RRF)
0,Jak zakończyć umowę?,✅ contract-general,✅ contract-general,✅ contract-general
1,Dokument HRB 80430,✅ hrb-80430,❌ hrb-80428,✅ hrb-80430
2,Problemy z połączeniem z bazą danych,❌ err-10530,✅ db-troubleshooting,❌ err-10530
3,Błąd ERR-10532,✅ err-10532,✅ err-10532,✅ err-10532
4,Warunki rozwiązania umowy nr 120/2026,✅ contract-120,✅ contract-120,✅ contract-120


**Co widzimy:** wyszukiwanie wektorowe myli niemal identyczne identyfikatory (`HRB 80428` vs `HRB 80430`), a BM25 trafia w nie bezbłędnie. Hybrid search naprawia zapytania o identyfikatory, ale dziedziczy też błąd BM25 dla *Problemy z połączeniem z bazą danych*: właściwy dokument mówi o *PostgreSQL* i *puli połączeń*, więc BM25 nie ma z nim mocnego dopasowania słów (a polska fleksja dokłada swoje: *bazą* ≠ *bazy*) i wybiera dokument, który dzieli z zapytaniem tylko popularne słowa. To zadanie dla kolejnego etapu – rerankingu.

## 2. Reranking z użyciem cross-encodera

Wyszukiwanie wektorowe, którego używaliśmy, to **bi-encoder**: zapytanie i każdy dokument są zamieniane na wektory *niezależnie*, a potem porównywane podobieństwem kosinusowym. To szybkie (wektory dokumentów liczymy wcześniej), ale zgrubne – model nigdy nie widzi zapytania i dokumentu razem.

**Cross-encoder** czyta parę `[zapytanie] + [dokument]` *wspólnie*, w jednym przebiegu transformera, i zwraca ocenę dopasowania. Jest dużo dokładniejszy, ale zbyt wolny, by przejść przez całą kolekcję. Stąd wzorzec dwuetapowy:

```
10 000 dokumentów
        │
        ▼
 Hybrid / Vector Search   ← szybko, wysoki recall
        │
        ▼
   30 kandydatów
        │
        ▼
     Reranker             ← wolno, wysoka precyzja
        │
        ▼
 5 najlepszych fragmentów
        │
        ▼
       LLM
```

Używamy `BAAI/bge-reranker-v2-m3` – otwartego, wielojęzycznego (także polski) rerankera – przez `sentence-transformers`. `HuggingFaceCrossEncoder` z LangChain ładuje go jako `sentence_transformers.CrossEncoder`, więc wczytujemy model raz i korzystamy z obu API.

In [10]:
from langchain_community.cross_encoders import HuggingFaceCrossEncoder
from sentence_transformers import CrossEncoder

cross_encoder = HuggingFaceCrossEncoder(model_name="BAAI/bge-reranker-v2-m3")
reranker: CrossEncoder = cross_encoder.client  # model sentence-transformers pod spodem

reranker.predict([
    ("Jaka jest stolica Francji?", "Paryż jest stolicą Francji."),
    ("Jaka jest stolica Francji?", "Francja słynie z serów."),
])

/home/mzar/anaconda3/envs/course-llm-agent-apps-with-langchain-and-langgraph/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 393/393 [00:00<00:00, 13779.18it/s]


array([9.9962819e-01, 5.5438164e-04], dtype=float32)

### Przed i po rerankingu

Hybrid search pomylił się dla zapytania *Problemy z połączeniem z bazą danych*. Bierzemy wszystkich kandydatów z hybrid search dla tego zapytania i pozwalamy cross-encoderowi zmienić ich kolejność.

In [11]:
query = "Problemy z połączeniem z bazą danych"
candidates = hybrid.invoke(query)

scores = reranker.predict([(query, d.page_content) for d in candidates])
before = {d.metadata["id"]: rank for rank, d in enumerate(candidates, start=1)}
reranked = sorted(zip(candidates, scores, strict=True), key=lambda pair: pair[1], reverse=True)

pd.DataFrame(
    {
        "pozycja po": rank,
        "pozycja przed": before[doc.metadata["id"]],
        "zmiana": f"{before[doc.metadata['id']] - rank:+d}",
        "wynik": round(float(score), 3),
        "id": doc.metadata["id"],
        "tekst": doc.page_content,
    }
    for rank, (doc, score) in enumerate(reranked, start=1)
)

,pozycja po,pozycja przed,zmiana,wynik,id,tekst
0,1,2,+1,0.02,db-troubleshooting,"Jeśli aplikacja nie może dostać się do PostgreSQL, sprawdź, czy pula połącze..."
1,2,4,+2,0.00,db-backup,Kopie zapasowe bazy danych wykonywane są co noc o 2:00 i przechowywane przez...
2,3,6,+3,0.00,err-10532,"ERR-10532: bramka płatności odrzuciła transakcję, ponieważ zapisany token ka..."
3,4,10,+6,0.00,err-10536,"ERR-10536: nie udało się odnowić subskrypcji, ponieważ plan został zarchiwiz..."
4,5,8,+3,0.00,err-10534,ERR-10534: waluta zamówienia nie jest obsługiwana przez konto sprzedawcy.
5,6,3,-3,0.00,err-10535,ERR-10535: adres rozliczeniowy nie zgadza się z kartą.
6,7,1,-6,0.00,err-10530,ERR-10530: bank odrzucił przelew z powodu braku środków.
7,8,5,-3,0.00,contract-120,Umowa nr 120/2026 z Nordic Logistics: rozwiązanie wymaga 90-dniowego wypowie...
8,9,9,+0,0.00,contract-general,Zakończenie współpracy: każda ze stron może wypowiedzieć umowę serwisową z 3...
9,10,7,-3,0.00,contract-121,Umowa nr 121/2026 z Baltic Foods: rozwiązanie wymaga 60-dniowego wypowiedzen...


### Reranking jako retriever LangChain

`ContextualCompressionRetriever` + `CrossEncoderReranker` zamieniają *dowolny* retriever w retriever „wyszukaj, potem przeszereguj”, który zwraca tylko `top_n` najlepszych dokumentów.

In [12]:
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import CrossEncoderReranker

hybrid_reranked = ContextualCompressionRetriever(
    base_retriever=hybrid,
    base_compressor=CrossEncoderReranker(model=cross_encoder, top_n=3),
)

for doc in hybrid_reranked.invoke(query):
    print(f"- [{doc.metadata['id']}] {doc.page_content}")

- [db-troubleshooting] Jeśli aplikacja nie może dostać się do PostgreSQL, sprawdź, czy pula połączeń nie jest wyczerpana i czy firewall przepuszcza ruch na porcie 5432.
- [db-backup] Kopie zapasowe bazy danych wykonywane są co noc o 2:00 i przechowywane przez 30 dni.
- [err-10532] ERR-10532: bramka płatności odrzuciła transakcję, ponieważ zapisany token karty wygasł. Poproś klienta o ponowne podanie danych karty.


In [13]:
compare({"Vector": vector, "Hybrid (RRF)": hybrid, "Hybrid + rerank": hybrid_reranked}, queries)

,zapytanie,Vector,Hybrid (RRF),Hybrid + rerank
0,Jak zakończyć umowę?,✅ contract-general,✅ contract-general,✅ contract-general
1,Dokument HRB 80430,❌ hrb-80428,✅ hrb-80430,✅ hrb-80430
2,Problemy z połączeniem z bazą danych,✅ db-troubleshooting,❌ err-10530,✅ db-troubleshooting
3,Błąd ERR-10532,✅ err-10532,✅ err-10532,✅ err-10532
4,Warunki rozwiązania umowy nr 120/2026,✅ contract-120,✅ contract-120,✅ contract-120


## 3. Nowoczesne strategie chunkingu

Retrieval może zwrócić tylko to, co wyprodukował chunker. Fragment, który przecina zdanie w połowie albo traci informację o tym, *z jakiego dokumentu i sekcji pochodzi*, nie zostanie znaleziony, niezależnie od jakości retrievera.

| Metoda | Zasada działania |
|---|---|
| Fixed-size chunking | Podział co określoną liczbę znaków lub tokenów |
| Recursive chunking | Uwzględnianie akapitów, zdań i separatorów |
| Semantic chunking | Podział na podstawie zmian znaczenia tekstu |
| Structure-aware chunking | Uwzględnianie nagłówków, tabel i struktury dokumentu |
| Parent-child retrieval | Wyszukiwanie małych fragmentów i zwracanie większego kontekstu |
| Contextual Retrieval | Wzbogacenie fragmentu o informacje dotyczące jego dokumentu źródłowego |

Naszym przykładowym dokumentem jest krótki raport roczny w Markdownie:

In [14]:
abc_report = """# ABC Corporation – Raport roczny 2025

## Rynek europejski

### Przychody
Przychody wzrosły o 18% względem poprzedniego roku. Wzrost wynikał głównie z nowych kontraktów korporacyjnych i wyższych cen subskrypcji.

### Zatrudnienie
Zespół urósł z 420 do 510 osób. Otwarto nowe biuro w Krakowie.

## Rynek azjatycki

### Przychody
Przychody wzrosły o 7% względem poprzedniego roku. Efekty kursowe obniżyły wynik o około dwa punkty procentowe.

### Zatrudnienie
Liczba pracowników utrzymała się na poziomie 150 osób. Rekrutacje wstrzymano w drugiej połowie roku.

## Perspektywy
Na 2026 rok zarząd oczekuje dwucyfrowego wzrostu. Główne ryzyka to rosnące koszty chmury i silniejsza konkurencja.
"""

In [15]:
def show(chunks, n: int = 4) -> None:
    for i, chunk in enumerate(chunks[:n], start=1):
        text = chunk.page_content if isinstance(chunk, Document) else chunk
        print(f"--- fragment {i} ({len(text)} znaków) ---\n{text}")
    print(f"... łącznie {len(chunks)} fragmentów")

### 3.1 Fixed-size chunking

Najprostsza strategia: cięcie co N znaków. Zwróć uwagę, jak rozrywane są słowa, zdania i sekcje.

In [16]:
from langchain_text_splitters import CharacterTextSplitter

fixed = CharacterTextSplitter(separator="", chunk_size=120, chunk_overlap=0)
show(fixed.split_text(abc_report))

--- fragment 1 (120 znaków) ---
# ABC Corporation – Raport roczny 2025

## Rynek europejski

### Przychody
Przychody wzrosły o 18% względem poprzedniego
--- fragment 2 (119 znaków) ---
roku. Wzrost wynikał głównie z nowych kontraktów korporacyjnych i wyższych cen subskrypcji.

### Zatrudnienie
Zespół ur
--- fragment 3 (120 znaków) ---
ósł z 420 do 510 osób. Otwarto nowe biuro w Krakowie.

## Rynek azjatycki

### Przychody
Przychody wzrosły o 7% względem
--- fragment 4 (119 znaków) ---
poprzedniego roku. Efekty kursowe obniżyły wynik o około dwa punkty procentowe.

### Zatrudnienie
Liczba pracowników ut
... łącznie 6 fragmentów


### 3.2 Recursive chunking

`RecursiveCharacterTextSplitter` próbuje separatorów po kolei – `\n\n`, `\n`, ` `, `""` – i sięga po drobniejszy tylko wtedy, gdy fragment wciąż jest za długi. To rozsądny wybór domyślny dla zwykłego tekstu. `from_language(Language.MARKDOWN)` / `Language.PYTHON` itd. dają separatory dopasowane do formatu.

In [17]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

recursive = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=30)
show(recursive.split_text(abc_report))

--- fragment 1 (59 znaków) ---
# ABC Corporation – Raport roczny 2025

## Rynek europejski
--- fragment 2 (172 znaków) ---
## Rynek europejski

### Przychody
Przychody wzrosły o 18% względem poprzedniego roku. Wzrost wynikał głównie z nowych kontraktów korporacyjnych i wyższych cen subskrypcji.
--- fragment 3 (99 znaków) ---
### Zatrudnienie
Zespół urósł z 420 do 510 osób. Otwarto nowe biuro w Krakowie.

## Rynek azjatycki
--- fragment 4 (145 znaków) ---
## Rynek azjatycki

### Przychody
Przychody wzrosły o 7% względem poprzedniego roku. Efekty kursowe obniżyły wynik o około dwa punkty procentowe.
... łącznie 6 fragmentów


### 3.3 Semantic chunking

Zamiast liczyć znaki patrzymy na **znaczenie**: zamieniamy każde zdanie na embedding i zaczynamy nowy fragment tam, gdzie dwa sąsiednie zdania są od siebie semantycznie odległe (tu: odległość powyżej 75. percentyla). Ten sam algorytm jest dostępny jako `SemanticChunker` w `langchain_experimental`; napisanie go ręcznie pokazuje, że nie ma w nim magii.

In [18]:
import numpy as np


def semantic_chunks(text: str, percentile: float = 75) -> list[str]:
    # usuń nagłówki Markdown i podziel tekst na zdania
    plain = " ".join(line for line in text.splitlines() if line and not line.startswith("#"))
    sentences = re.split(r"(?<=[.!?])\s+", plain)
    vectors = np.array(embeddings.embed_documents(sentences))
    vectors /= np.linalg.norm(vectors, axis=1, keepdims=True)
    distances = 1 - np.sum(vectors[:-1] * vectors[1:], axis=1)  # odległość kosinusowa do kolejnego zdania
    threshold = np.percentile(distances, percentile)

    chunks, current = [], [sentences[0]]
    for sentence, distance in zip(sentences[1:], distances, strict=True):
        if distance > threshold:
            chunks.append(" ".join(current))
            current = []
        current.append(sentence)
    return chunks + [" ".join(current)]


semantic = semantic_chunks(abc_report)
assert " ".join(semantic).count(".") == abc_report.count(".")  # żadne zdanie nie zginęło
show(semantic, n=10)

--- fragment 1 (200 znaków) ---
Przychody wzrosły o 18% względem poprzedniego roku. Wzrost wynikał głównie z nowych kontraktów korporacyjnych i wyższych cen subskrypcji. Zespół urósł z 420 do 510 osób. Otwarto nowe biuro w Krakowie.
--- fragment 2 (111 znaków) ---
Przychody wzrosły o 7% względem poprzedniego roku. Efekty kursowe obniżyły wynik o około dwa punkty procentowe.
--- fragment 3 (215 znaków) ---
Liczba pracowników utrzymała się na poziomie 150 osób. Rekrutacje wstrzymano w drugiej połowie roku. Na 2026 rok zarząd oczekuje dwucyfrowego wzrostu. Główne ryzyka to rosnące koszty chmury i silniejsza konkurencja.
... łącznie 3 fragmentów


### 3.4 Structure-aware chunking

Dokumenty mają już strukturę – nagłówki, sekcje, tabele. `MarkdownHeaderTextSplitter` dzieli tekst po nagłówkach i zapisuje ścieżkę nagłówków w **metadanych**, dzięki czemu każdy fragment pamięta, skąd pochodzi (dla HTML jest `HTMLHeaderTextSplitter`).

In [19]:
from langchain_text_splitters import MarkdownHeaderTextSplitter

markdown_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=[("#", "document"), ("##", "section"), ("###", "subsection")]
)
sections = markdown_splitter.split_text(abc_report)

for doc in sections[:3]:
    print(doc.metadata, "\n   ", doc.page_content, "\n")

{'document': 'ABC Corporation – Raport roczny 2025', 'section': 'Rynek europejski', 'subsection': 'Przychody'} 
    Przychody wzrosły o 18% względem poprzedniego roku. Wzrost wynikał głównie z nowych kontraktów korporacyjnych i wyższych cen subskrypcji. 

{'document': 'ABC Corporation – Raport roczny 2025', 'section': 'Rynek europejski', 'subsection': 'Zatrudnienie'} 
    Zespół urósł z 420 do 510 osób. Otwarto nowe biuro w Krakowie. 

{'document': 'ABC Corporation – Raport roczny 2025', 'section': 'Rynek azjatycki', 'subsection': 'Przychody'} 
    Przychody wzrosły o 7% względem poprzedniego roku. Efekty kursowe obniżyły wynik o około dwa punkty procentowe. 



### 3.5 Parent-child retrieval

Małe fragmenty są **precyzyjne w wyszukiwaniu** (jedna myśl na embedding), duże są **użyteczne w czytaniu** (LLM dostaje wystarczający kontekst). Parent-child retrieval łączy obie zalety: indeksuje małe fragmenty *dzieci*, ale zwraca ich *rodzica* (tu: całą sekcję o danym rynku).

In [20]:
from langchain_classic.retrievers import ParentDocumentRetriever
from langchain_classic.storage import InMemoryStore
from langchain_core.vectorstores import InMemoryVectorStore

child_vectorstore = InMemoryVectorStore(embeddings)
parent_retriever = ParentDocumentRetriever(
    vectorstore=child_vectorstore,       # małe fragmenty -> embeddingi
    docstore=InMemoryStore(),            # dokumenty-rodzice według id
    child_splitter=RecursiveCharacterTextSplitter(chunk_size=80, chunk_overlap=0),
    search_kwargs={"k": 1},
)
# rodzice: całe sekcje "##" o rynkach (Przychody + Zatrudnienie razem)
parents = MarkdownHeaderTextSplitter(headers_to_split_on=[("#", "document"), ("##", "section")]).split_text(abc_report)
parent_retriever.add_documents(parents)

question = "Gdzie ABC otworzyło nowe biuro?"
print("DZIECKO znalezione przez wyszukiwanie wektorowe:\n   ", child_vectorstore.similarity_search(question, k=1)[0].page_content)
print("RODZIC przekazany do LLM:\n   ", parent_retriever.invoke(question)[0].page_content)

DZIECKO znalezione przez wyszukiwanie wektorowe:
    ### Zatrudnienie
Zespół urósł z 420 do 510 osób. Otwarto nowe biuro w Krakowie.
RODZIC przekazany do LLM:
    ### Przychody
Przychody wzrosły o 18% względem poprzedniego roku. Wzrost wynikał głównie z nowych kontraktów korporacyjnych i wyższych cen subskrypcji.  
### Zatrudnienie
Zespół urósł z 420 do 510 osób. Otwarto nowe biuro w Krakowie.


### 3.6 Contextual Retrieval

Spójrz na dwa fragmenty *Przychody* z powyższego raportu. Wyjęty z dokumentu fragment

> Przychody wzrosły o 18% względem poprzedniego roku.

nie mówi, **której firmy, którego rynku ani którego roku** dotyczy. Dodaj drugi raport z identycznym zdaniem, a ani BM25, ani embeddingi nie odróżnią tych fragmentów.

[Contextual Retrieval](https://www.anthropic.com/news/contextual-retrieval) (Anthropic) rozwiązuje to na etapie indeksowania: LLM czyta cały dokument i pisze krótki kontekst, który **dokleja się na początku każdego fragmentu** przed policzeniem embeddingu i indeksowaniem w BM25:

> Raport finansowy firmy ABC za 2025 rok, sekcja dotycząca rynku europejskiego: przychody wzrosły o 18% względem poprzedniego roku.

Drugi fragment ma znacznie większą szansę zostać znaleziony dla odpowiedniego zapytania. W eksperymentach Anthropic kontekstowe embeddingi + kontekstowy BM25 zmniejszyły liczbę nieudanych wyszukiwań o 49%, a w połączeniu z rerankingiem o 67%. Prompt caching sprawia, że jest to tanie: cały dokument trafia do cache raz i jest używany ponownie dla każdego fragmentu.

Zbudujmy mały korpus z **trzema raportami**, których fragmenty o przychodach wyglądają prawie tak samo:

In [21]:
xyz_report = """# XYZ Retail – Raport roczny 2025

## Rynek europejski

### Przychody
Przychody wzrosły o 11% względem poprzedniego roku. Wzrost wynikał głównie ze sprzedaży internetowej.

### Zatrudnienie
Zespół zmniejszył się z 900 do 860 osób po zamknięciu dwóch sklepów.

## Perspektywy
Na 2026 rok zarząd oczekuje stabilnych przychodów i niższych kosztów logistyki.
"""

abc_report_2024 = """# ABC Corporation – Raport roczny 2024

## Rynek europejski

### Przychody
Przychody wzrosły o 9% względem poprzedniego roku. Wzrost wynikał głównie z obecnych klientów.

## Rynek azjatycki

### Przychody
Przychody wzrosły o 12% względem poprzedniego roku. Wzrost wynikał głównie z nowego biura w Singapurze.
"""

reports = {"ABC-2025": abc_report, "ABC-2024": abc_report_2024, "XYZ-2025": xyz_report}

# fragmenty structure-aware BEZ nagłówków w treści – realistyczna, „naiwna” konfiguracja
plain_chunks = []
for report_id, report in reports.items():
    for i, section in enumerate(markdown_splitter.split_text(report)):
        section.metadata |= {"id": f"{report_id}-{i}", "full_document": report}
        plain_chunks.append(section)

len(plain_chunks)

10

Teraz krok kontekstualizacji – prompt jest wzorowany na tym opublikowanym przez Anthropic. `llm.batch` wysyła wszystkie fragmenty równolegle.

In [22]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

context_prompt = ChatPromptTemplate.from_template(
    """<document>
{document}
</document>
Oto fragment, który chcemy umiejscowić w całym dokumencie:
<chunk>
{chunk}
</chunk>
Podaj krótki, zwięzły kontekst (jedno zdanie: dokument, firma, rok, sekcja), który umiejscawia ten fragment
w całym dokumencie, aby ułatwić jego wyszukiwanie.
Odpowiedz wyłącznie tym kontekstem, po polsku, bez niczego więcej."""
)
contextualizer = context_prompt | llm | StrOutputParser()

contexts = contextualizer.batch(
    [{"document": c.metadata["full_document"], "chunk": c.page_content} for c in plain_chunks]
)
contextual_chunks = [
    Document(page_content=f"{ctx}\n{c.page_content}", metadata=c.metadata)
    for ctx, c in zip(contexts, plain_chunks, strict=True)
]

print("PRZED:\n", plain_chunks[0].page_content)
print("\nPO:\n", contextual_chunks[0].page_content)

PRZED:
 Przychody wzrosły o 18% względem poprzedniego roku. Wzrost wynikał głównie z nowych kontraktów korporacyjnych i wyższych cen subskrypcji.

PO:
 Raport roczny ABC Corporation za 2025 rok, sekcja „Rynek europejski” – podsekcja „Przychody”.
Przychody wzrosły o 18% względem poprzedniego roku. Wzrost wynikał głównie z nowych kontraktów korporacyjnych i wyższych cen subskrypcji.


Teraz test. Przeszukujemy oba indeksy tym samym pytaniem i pozwalamy LLM odpowiedzieć **wyłącznie na podstawie znalezionych fragmentów**. Zwykły fragment *„Przychody wzrosły o 18%…”* może nawet zostać znaleziony, ale ani retriever, ani LLM nie wiedzą, czy dotyczy ABC czy XYZ, Europy czy Azji, 2024 czy 2025 roku.

In [23]:
answer_prompt = ChatPromptTemplate.from_template(
    "Odpowiadaj wyłącznie na podstawie KONTEKSTU. Jeśli KONTEKST nie mówi tego jednoznacznie, odpowiedz „Nie wiem”.\n\n"
    "KONTEKST:\n{context}\n\nPYTANIE: {question}"
)
question = "O ile wzrosły przychody ABC w Europie w 2025 roku?"

for name, chunks in {"ZWYKŁE fragmenty": plain_chunks, "fragmenty Z KONTEKSTEM": contextual_chunks}.items():
    found = FAISS.from_documents(chunks, embeddings).similarity_search(question, k=3)
    context = "\n\n".join(d.page_content for d in found)
    print(f"=== {name} ===")
    for d in found:
        print(f"  [{d.metadata['id']}] {d.page_content[:120]}...")
    print("  LLM:", (answer_prompt | llm | StrOutputParser()).invoke({"context": context, "question": question}), "\n")

=== ZWYKŁE fragmenty ===
  [ABC-2025-0] Przychody wzrosły o 18% względem poprzedniego roku. Wzrost wynikał głównie z nowych kontraktów korporacyjnych i wyższych...
  [ABC-2024-0] Przychody wzrosły o 9% względem poprzedniego roku. Wzrost wynikał głównie z obecnych klientów....
  [ABC-2025-4] Na 2026 rok zarząd oczekuje dwucyfrowego wzrostu. Główne ryzyka to rosnące koszty chmury i silniejsza konkurencja....
  LLM: Nie wiem. 

=== fragmenty Z KONTEKSTEM ===
  [ABC-2025-0] Raport roczny ABC Corporation za 2025 rok, sekcja „Rynek europejski” – podsekcja „Przychody”.
Przychody wzrosły o 18% wz...
  [ABC-2024-0] Fragment pochodzi z raportu rocznego ABC Corporation za 2024 rok, z sekcji „Rynek europejski – Przychody”.
Przychody wzr...
  [ABC-2025-4] Raport roczny ABC Corporation za 2025 rok, sekcja „Perspektywy”.
Na 2026 rok zarząd oczekuje dwucyfrowego wzrostu. Główn...
  LLM: Przychody ABC w Europie wzrosły w 2025 roku o 18% względem poprzedniego roku. 



💡 W tak małym korpusie słowa takie jak *ABC* czy *2025* występują w większości fragmentów z kontekstem, więc ich IDF w BM25 jest bliskie zera. W prawdziwych kolekcjach z tysiącami dokumentów te same słowa kontekstu stają się silnymi sygnałami leksykalnymi – dlatego Anthropic zaleca łączenie **kontekstowych embeddingów + kontekstowego BM25 + rerankingu**.

## 4. Wszystko razem – RAG w stylu produkcyjnym

```
dokumenty ──► fragmenty structure-aware ──► Contextual Retrieval ──► BM25 + FAISS
                                                                         │
pytanie ────────────────────────────────────────────────► hybrid search (RRF)
                                                                         │
                                                          reranking cross-encoderem (top 3)
                                                                         │
                                                                        LLM
```

In [24]:
from langchain_core.runnables import RunnablePassthrough


def build_hybrid(chunks: list[Document], k: int) -> EnsembleRetriever:
    return EnsembleRetriever(
        retrievers=[
            BM25Retriever.from_documents(chunks, preprocess_func=tokenize, k=k),
            FAISS.from_documents(chunks, embeddings).as_retriever(search_kwargs={"k": k}),
        ],
        weights=[0.5, 0.5],
    )


retriever = ContextualCompressionRetriever(
    base_retriever=build_hybrid(contextual_chunks, k=10),  # recall: do 20 kandydatów
    base_compressor=CrossEncoderReranker(model=cross_encoder, top_n=3),  # precyzja
)

rag_prompt = ChatPromptTemplate.from_messages([
    ("system", "Odpowiadaj wyłącznie na podstawie KONTEKSTU. Podaj firmę i sekcję. Jeśli odpowiedzi tam nie ma, powiedz, że nie wiesz.\n\nKONTEKST:\n{context}"),
    ("user", "{question}"),
])


def format_docs(docs: list[Document]) -> str:
    return "\n\n".join(d.page_content for d in docs)


rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
)

for question in [
    "O ile wzrosły przychody ABC w Europie w 2025 roku i dlaczego?",
    "Porównaj wzrost przychodów ABC i XYZ na rynku europejskim w 2025 roku.",
    "Jak zmienił się wzrost przychodów ABC w Europie między 2024 a 2025 rokiem?",
    "Jakie były przychody ABC w Afryce?",
]:
    print(f"P: {question}\nO: {rag_chain.invoke(question)}\n")

P: O ile wzrosły przychody ABC w Europie w 2025 roku i dlaczego?
O: **ABC Corporation — „Rynek europejski – Przychody” (raport roczny za 2025 rok):** Przychody wzrosły o 18% rok do roku, głównie dzięki nowym kontraktom korporacyjnym i wyższym cenom subskrypcji.

P: Porównaj wzrost przychodów ABC i XYZ na rynku europejskim w 2025 roku.
O: W 2025 roku na rynku europejskim przychody ABC Corporation wzrosły o 18%, a XYZ Retail o 11% — wzrost ABC był większy o 7 punktów procentowych.

- **ABC Corporation — „Rynek europejski – Przychody”**: wzrost wynikał głównie z nowych kontraktów korporacyjnych i wyższych cen subskrypcji.
- **XYZ Retail — „Rynek europejski – Przychody”**: wzrost wynikał głównie ze sprzedaży internetowej.

P: Jak zmienił się wzrost przychodów ABC w Europie między 2024 a 2025 rokiem?
O: ABC Corporation, sekcja „Rynek europejski – Przychody”: wzrost przychodów zwiększył się z 9% w 2024 roku do 18% w 2025 roku, czyli o 9 punktów procentowych.

P: Jakie były przychody ABC w Af